# 041 Blend Add039 + GPU040 Check

Blend/correlation check after adding 039 CatBoost variant and 040 GPU LogReg stacker.

In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260611_041_blend_add039_gpu040_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

# Primary location. The loader below also scans /kaggle/input if a file is not found here.
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
INPUT_ROOT = Path("/kaggle/input")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {"key": "015_realmlp_seed42", "exp_id": "exp_20260605_015_shared001_realmlp_as_is", "family": "RealMLP", "role": "stable_single_realmlp_backup", "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy", "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy", "cv": 0.9681693449222352, "public_lb": 0.96977},
    {"key": "017_realmlp_bias", "exp_id": "exp_20260606_017_bias_search_on_015_realmlp", "family": "RealMLP", "role": "previous_best_realmlp_bias_backup", "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy", "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy", "cv": 0.9683022653955233, "public_lb": 0.96985},
    {"key": "018_xgb_shared003", "exp_id": "exp_20260606_018_shared003_xgb_as_is", "family": "XGBoost", "role": "effective_blend_material", "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy", "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy", "cv": 0.965207986418628, "public_lb": 0.96578},
    {"key": "019_blend_best", "exp_id": "exp_20260607_019_blend_add018_xgb_check", "family": "Blend", "role": "previous_public_confirmed_best", "oof": "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy", "pred": "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy", "cv": 0.968872315017554, "public_lb": 0.97000},
    {"key": "020_blend_bias", "exp_id": "exp_20260607_020_bias_search_on_019_best_blend", "family": "Blend", "role": "cv_trusted_attack_reference", "oof": "oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy", "pred": "pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy", "cv": 0.9692240443617589, "public_lb": 0.96969},
    {"key": "024_seed_ensemble_blend", "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check", "family": "Blend", "role": "seed_ensemble_blend_reference", "oof": "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy", "pred": "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy", "cv": 0.9694803109983217, "public_lb": 0.96958},
    {"key": "026_realmlp_v5", "exp_id": "exp_20260608_026_realmlp_v5_as_is", "family": "RealMLP", "role": "realmlp_v5_single_model_candidate", "oof": "oof_exp_20260608_026_realmlp_v5_as_is_proba.npy", "pred": "pred_exp_20260608_026_realmlp_v5_as_is_proba.npy", "cv": 0.9690389777981231, "public_lb": 0.96979},
    {"key": "027_blend_add026", "exp_id": "exp_20260608_027_blend_add026_realmlp_v5_check", "family": "Blend", "role": "previous_cv_trusted_slot", "oof": "oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy", "pred": "pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy", "cv": 0.9695425457477898, "public_lb": 0.96975},
    {"key": "028_cat_v3", "exp_id": "exp_20260608_028_cat_v3_as_is", "family": "CatBoost", "role": "catboost_v3_family_aux_material", "oof": "oof_exp_20260608_028_cat_v3_as_is_proba.npy", "pred": "pred_exp_20260608_028_cat_v3_as_is_proba.npy", "cv": 0.9688146470512534, "public_lb": 0.96969},
    {"key": "029_blend_add028", "exp_id": "exp_20260608_029_blend_add028_cat_v3_check", "family": "Blend", "role": "previous_best_backup", "oof": "oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy", "pred": "pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy", "cv": 0.9700023026377228, "public_lb": 0.970036},
    {"key": "030_ovr_xgb", "exp_id": "exp_20260608_030_ovr_xgb_as_is", "family": "XGBoost", "role": "low_weight_auxiliary_xgb_ovr_material", "oof": "oof_exp_20260608_030_ovr_xgb_as_is_proba.npy", "pred": "pred_exp_20260608_030_ovr_xgb_as_is_proba.npy", "cv": 0.9609971296999271, "public_lb": 0.96118},
    {"key": "031_blend_add030", "exp_id": "exp_20260608_031_blend_add030_ovr_xgb_check", "family": "Blend", "role": "public_confirmed_current_best_before_033", "oof": "oof_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy", "pred": "pred_exp_20260608_031_blend_add030_ovr_xgb_check_best_blend_proba.npy", "cv": 0.9700333620193362, "public_lb": 0.97043},
    {"key": "032_ovr_tabm", "exp_id": "exp_20260609_032_ovr_tabm_as_is", "family": "TabM", "role": "tabm_ovr_family_aux_material", "oof": "oof_exp_20260609_032_ovr_tabm_as_is_proba.npy", "pred": "pred_exp_20260609_032_ovr_tabm_as_is_proba.npy", "cv": 0.9610105651284856, "public_lb": 0.96106, "tuned_cv": 0.9686168281485955, "public_lb_biased": 0.96895},
    {"key": "033_blend_add032", "exp_id": "exp_20260609_033_blend_add032_tabm_check", "family": "Blend", "role": "previous_public_confirmed_best_backup", "oof": "oof_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy", "pred": "pred_exp_20260609_033_blend_add032_tabm_check_best_blend_proba.npy", "cv": 0.9700400336552478, "public_lb": 0.97043},
    {"key": "034_gpu_logreg_default", "exp_id": "exp_20260609_034_gpu_logreg_stacker_own", "family": "GPU_LogisticRegression_Stacker", "role": "stacker_backup_default", "oof": "oof_exp_20260609_034_gpu_logreg_stacker_own_proba.npy", "pred": "pred_exp_20260609_034_gpu_logreg_stacker_own_proba.npy", "cv": 0.9700373069292101, "public_lb": 0.97022, "raw_cv": 0.9699389938897909, "public_lb_raw": 0.97027},
    {"key": "035_shared001_updated", "exp_id": "exp_20260610_035_shared001_updated_realmlp_pytorch_as_is", "family": "RealMLP", "role": "updated_shared001_realmlp_aux_material", "oof": "oof_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy", "pred": "pred_exp_20260610_035_shared001_updated_realmlp_pytorch_as_is_proba.npy", "cv": 0.9687410900866934, "public_lb": 0.97012},
    {"key": "036_gpu_logreg_add035", "exp_id": "exp_20260610_036_gpu_logreg_stacker_add035_own", "family": "GPU_LogisticRegression_Stacker", "role": "improved_stacker_backup_add035", "oof": "oof_exp_20260610_036_gpu_logreg_stacker_add035_own_proba.npy", "pred": "pred_exp_20260610_036_gpu_logreg_stacker_add035_own_proba.npy", "cv": 0.9700674063284508, "public_lb": 0.97037},
    {"key": "036_blend_add035", "exp_id": "exp_20260610_036_blend_add035_shared001_check", "family": "Blend", "role": "cv_best_but_public_weak_add035_blend", "oof": "oof_exp_20260610_036_blend_add035_shared001_check_best_blend_proba.npy", "pred": "pred_exp_20260610_036_blend_add035_shared001_check_best_blend_proba.npy", "cv": 0.9700727843277738, "public_lb": 0.97023},
    {"key": "037_tabicl_v2", "exp_id": "exp_20260610_037_tabicl_v2_as_is", "family": "TabICL", "role": "weak_tabicl_family_material_optional_corr", "oof": "oof_exp_20260610_037_tabicl_v2_as_is_proba.npy", "pred": "pred_exp_20260610_037_tabicl_v2_as_is_proba.npy", "cv": 0.9580770153777599, "public_lb": 0.95920},
    {"key": "038_gpu_logreg_add037", "exp_id": "exp_20260610_038_gpu_logreg_stacker_add037_own", "family": "GPU_LogisticRegression_Stacker", "role": "previous_stacker_best_before_040", "oof": "oof_exp_20260610_038_gpu_logreg_stacker_add037_own_proba.npy", "pred": "pred_exp_20260610_038_gpu_logreg_stacker_add037_own_proba.npy", "cv": 0.9701353365798572, "public_lb": 0.97060},
    {"key": "039_cat_v3_seed2026_qbin_variant", "exp_id": "exp_20260611_039_cat_v3_seed2026_qbin_variant", "family": "CatBoost", "role": "catboost_v3_lower_correlation_variant_material", "oof": "oof_exp_20260611_039_cat_v3_seed2026_qbin_variant_proba.npy", "pred": "pred_exp_20260611_039_cat_v3_seed2026_qbin_variant_proba.npy", "cv": 0.9687053023092482, "public_lb": 0.96984},
    {"key": "040_gpu_logreg_add039", "exp_id": "exp_20260611_040_gpu_logreg_stacker_add039_own", "family": "GPU_LogisticRegression_Stacker", "role": "current_best_after_add039", "oof": "oof_exp_20260611_040_gpu_logreg_stacker_add039_own_proba.npy", "pred": "pred_exp_20260611_040_gpu_logreg_stacker_add039_own_proba.npy", "cv": 0.9702017877238539, "public_lb": 0.97069},
]

TRACK_KEYS = {
    "040": "040_gpu_logreg_add039",
    "039": "039_cat_v3_seed2026_qbin_variant",
    "038": "038_gpu_logreg_add037",
    "037": "037_tabicl_v2",
    "036_gpu": "036_gpu_logreg_add035",
    "036_blend": "036_blend_add035",
    "035": "035_shared001_updated",
    "034": "034_gpu_logreg_default",
    "033": "033_blend_add032",
    "032": "032_ovr_tabm",
    "031": "031_blend_add030",
    "030": "030_ovr_xgb",
    "029": "029_blend_add028",
    "028": "028_cat_v3",
}

FOCUS_MODEL_KEYS = {
    "040": "040_gpu_logreg_add039",
    "039": "039_cat_v3_seed2026_qbin_variant",
    "038": "038_gpu_logreg_add037",
    "037": "037_tabicl_v2",
    "036_gpu": "036_gpu_logreg_add035",
    "036_blend": "036_blend_add035",
    "033": "033_blend_add032",
    "031": "031_blend_add030",
    "028": "028_cat_v3",
}

# 041 should answer:
# - Does 039 help simple probability blending after it improved the GPU LogReg stacker?
# - Can 040 be improved by direct safe blending with 033/038/039/036 variants?
# - Which candidates should remain as final submission slots or backups?
BLEND_SETS = {
    # Singles / current references
    "A_040_only": ["040_gpu_logreg_add039"],
    "B_038_only": ["038_gpu_logreg_add037"],
    "C_033_only": ["033_blend_add032"],
    "D_036_gpu_only": ["036_gpu_logreg_add035"],
    "E_036_blend_only": ["036_blend_add035"],
    "F_039_only": ["039_cat_v3_seed2026_qbin_variant"],
    "G_028_only": ["028_cat_v3"],
    "H_037_only": ["037_tabicl_v2"],
    "I_031_only": ["031_blend_add030"],

    # Direct add039 / add040 checks
    "J_040_033": ["040_gpu_logreg_add039", "033_blend_add032"],
    "K_040_038": ["040_gpu_logreg_add039", "038_gpu_logreg_add037"],
    "L_040_039": ["040_gpu_logreg_add039", "039_cat_v3_seed2026_qbin_variant"],
    "M_040_028": ["040_gpu_logreg_add039", "028_cat_v3"],
    "N_040_036gpu": ["040_gpu_logreg_add039", "036_gpu_logreg_add035"],
    "O_040_036blend": ["040_gpu_logreg_add039", "036_blend_add035"],
    "P_040_037": ["040_gpu_logreg_add039", "037_tabicl_v2"],
    "Q_033_039": ["033_blend_add032", "039_cat_v3_seed2026_qbin_variant"],
    "R_038_039": ["038_gpu_logreg_add037", "039_cat_v3_seed2026_qbin_variant"],
    "S_036gpu_039": ["036_gpu_logreg_add035", "039_cat_v3_seed2026_qbin_variant"],

    # Core current-best blend checks
    "T_040_033_039": ["040_gpu_logreg_add039", "033_blend_add032", "039_cat_v3_seed2026_qbin_variant"],
    "U_040_038_033": ["040_gpu_logreg_add039", "038_gpu_logreg_add037", "033_blend_add032"],
    "V_040_038_039": ["040_gpu_logreg_add039", "038_gpu_logreg_add037", "039_cat_v3_seed2026_qbin_variant"],
    "W_040_033_038_039": ["040_gpu_logreg_add039", "033_blend_add032", "038_gpu_logreg_add037", "039_cat_v3_seed2026_qbin_variant"],
    "X_040_033_036gpu_036blend_039": ["040_gpu_logreg_add039", "033_blend_add032", "036_gpu_logreg_add035", "036_blend_add035", "039_cat_v3_seed2026_qbin_variant"],

    # Component diagnostics. Interpret weights carefully because composite blends and components coexist.
    "Y_027_026_028_039_030_032_035_037": ["027_blend_add026", "026_realmlp_v5", "028_cat_v3", "039_cat_v3_seed2026_qbin_variant", "030_ovr_xgb", "032_ovr_tabm", "035_shared001_updated", "037_tabicl_v2"],
    "Z1_040_033_038_036gpu_036blend_039_037": ["040_gpu_logreg_add039", "033_blend_add032", "038_gpu_logreg_add037", "036_gpu_logreg_add035", "036_blend_add035", "039_cat_v3_seed2026_qbin_variant", "037_tabicl_v2"],
    "Z2_040_033_038_034_031_029_028_039": ["040_gpu_logreg_add039", "033_blend_add032", "038_gpu_logreg_add037", "034_gpu_logreg_default", "031_blend_add030", "029_blend_add028", "028_cat_v3", "039_cat_v3_seed2026_qbin_variant"],
    "Z3_core_no_040_033_038_036gpu_039": ["033_blend_add032", "038_gpu_logreg_add037", "036_gpu_logreg_add035", "039_cat_v3_seed2026_qbin_variant"],
    "Z_full_015_017_018_019_020_024_026_027_028_029_030_031_032_033_034_035_036gpu_036blend_037_038_039_040": [
        "015_realmlp_seed42", "017_realmlp_bias", "018_xgb_shared003",
        "019_blend_best", "020_blend_bias", "024_seed_ensemble_blend",
        "026_realmlp_v5", "027_blend_add026", "028_cat_v3",
        "029_blend_add028", "030_ovr_xgb", "031_blend_add030",
        "032_ovr_tabm", "033_blend_add032", "034_gpu_logreg_default",
        "035_shared001_updated", "036_gpu_logreg_add035", "036_blend_add035",
        "037_tabicl_v2", "038_gpu_logreg_add037", "039_cat_v3_seed2026_qbin_variant", "040_gpu_logreg_add039",
    ],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)
print("n_models:", len(MODEL_SPECS))
print("n_blend_sets:", len(BLEND_SETS))


EXP_ID: exp_20260611_041_blend_add039_gpu040_check
OUTDIR: /kaggle/working/exp_20260611_041_blend_add039_gpu040_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files
n_models: 22
n_blend_sets: 29


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def resolve_npy(filename):
    """Resolve an npy path. Prefer NPY_DIR, then scan /kaggle/input.

    This is intentionally more robust than 027 because 028 may be uploaded as a
    separate dataset from older npy files.
    """
    p = NPY_DIR / filename
    if p.exists():
        return p
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] multiple matches for {filename}. Using first:")
        for m in matches[:10]:
            print("  ", m)
        return matches[0]
    raise FileNotFoundError(f"Missing npy file: {filename}. Checked {NPY_DIR} and scanned {INPUT_ROOT}.")

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape)
    assert np.isfinite(arr).all(), name
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-4), name
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit":
            mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur_score = balanced_accuracy_score(y_true, normalize_rows(sum(w[i] * probas[i] for i in range(n))).argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score, best_w = cur_score, w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w /= cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score, best_w = score, cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score, w = best_score, best_w
                improved = True
                hist.append({"iter": len(hist), "step": step, "score": float(cur_score), "weights": w.copy().tolist()})
    return w, float(cur_score), hist

def nm_softmax_weights(y_true, probas, n_restarts=5, maxiter=2500):
    probas = [np.asarray(p, dtype=np.float64) for p in probas]
    n = len(probas)

    def eval_from_theta(theta):
        w = np.exp(theta - np.max(theta))
        w = w / w.sum()
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return balanced_accuracy_score(y_true, p.argmax(axis=1)), w

    def objective(theta):
        score, _ = eval_from_theta(theta)
        return -score

    inits = [np.zeros(n)]
    rng = np.random.default_rng(SEED)
    for _ in range(n_restarts - 1):
        inits.append(rng.normal(0, 0.25, size=n))

    best_score, best_w, best_res = -1, None, None
    for init in inits:
        res = minimize(objective, init, method="Nelder-Mead", options={"maxiter": maxiter, "xatol": 1e-8, "fatol": 1e-12})
        score, w = eval_from_theta(res.x)
        if score > best_score:
            best_score, best_w, best_res = score, w, res
    return best_w, float(best_score), best_res

def disagreement_and_error_overlap(y_true, p1, p2):
    pred1 = p1.argmax(axis=1)
    pred2 = p2.argmax(axis=1)
    wrong1 = pred1 != y_true
    wrong2 = pred2 != y_true
    both_wrong = wrong1 & wrong2
    either_wrong = wrong1 | wrong2
    return {
        "disagreement_rate": float((pred1 != pred2).mean()),
        "wrong_rate_left": float(wrong1.mean()),
        "wrong_rate_right": float(wrong2.mean()),
        "both_wrong_rate": float(both_wrong.mean()),
        "error_overlap_jaccard": float(both_wrong.sum() / max(either_wrong.sum(), 1)),
        "left_wrong_right_correct_rate": float((wrong1 & ~wrong2).mean()),
        "right_wrong_left_correct_rate": float((wrong2 & ~wrong1).mean()),
    }


In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = resolve_npy(spec["oof"])
    pred_path = resolve_npy(spec["pred"])
    oof_arr = np.load(oof_path)
    pred_arr = np.load(pred_path)
    validate_proba(f"oof:{key}", oof_arr, len(train), n_classes)
    validate_proba(f"pred:{key}", pred_arr, len(test), n_classes)
    oof[key] = oof_arr.astype(np.float64)
    pred[key] = pred_arr.astype(np.float64)
    s = spec.copy()
    s["resolved_oof_path"] = str(oof_path)
    s["resolved_pred_path"] = str(pred_path)
    resolved_specs.append(s)
    print(f"loaded {key}: {oof_arr.shape} / {pred_arr.shape}")

model_keys = [s["key"] for s in resolved_specs]
print("class_names:", class_names)
print("loaded keys:", model_keys)


loaded 015_realmlp_seed42: (577347, 3) / (247435, 3)
loaded 017_realmlp_bias: (577347, 3) / (247435, 3)
loaded 018_xgb_shared003: (577347, 3) / (247435, 3)
loaded 019_blend_best: (577347, 3) / (247435, 3)
loaded 020_blend_bias: (577347, 3) / (247435, 3)
loaded 024_seed_ensemble_blend: (577347, 3) / (247435, 3)
loaded 026_realmlp_v5: (577347, 3) / (247435, 3)
loaded 027_blend_add026: (577347, 3) / (247435, 3)
loaded 028_cat_v3: (577347, 3) / (247435, 3)
loaded 029_blend_add028: (577347, 3) / (247435, 3)
loaded 030_ovr_xgb: (577347, 3) / (247435, 3)
loaded 031_blend_add030: (577347, 3) / (247435, 3)
loaded 032_ovr_tabm: (577347, 3) / (247435, 3)
loaded 033_blend_add032: (577347, 3) / (247435, 3)
loaded 034_gpu_logreg_default: (577347, 3) / (247435, 3)
loaded 035_shared001_updated: (577347, 3) / (247435, 3)
loaded 036_gpu_logreg_add035: (577347, 3) / (247435, 3)
loaded 036_blend_add035: (577347, 3) / (247435, 3)
loaded 037_tabicl_v2: (577347, 3) / (247435, 3)
loaded 038_gpu_logreg_add037:

In [4]:
# ============================================================
# 3. Individual scores and pairwise diagnostics
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "public_lb_biased": spec.get("public_lb_biased", np.nan),
        "tuned_cv": spec.get("tuned_cv", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False).reset_index(drop=True)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

pair_rows = []
for a, b in combinations(model_keys, 2):
    diag = disagreement_and_error_overlap(y, oof[a], oof[b])
    pair_rows.append({
        "left": a,
        "right": b,
        "pearson_flat_proba": corr_pearson(oof[a], oof[b]),
        "spearman_flat_proba": corr_spearman(oof[a], oof[b]),
        **diag,
    })

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df = pairwise_df.sort_values(["pearson_flat_proba", "error_overlap_jaccard"], ascending=[True, True]).reset_index(drop=True)
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

# Focus tables for newly added / current reference models.
def focus_pairwise(key, filename):
    df = pairwise_df[(pairwise_df["left"] == key) | (pairwise_df["right"] == key)].copy()
    df = df.sort_values("pearson_flat_proba").reset_index(drop=True)
    display(df)
    df.to_csv(OUTDIR / filename, index=False)
    return df

focus_pairwise_tables = {}
for short_name, key in FOCUS_MODEL_KEYS.items():
    if key in model_keys:
        focus_pairwise_tables[short_name] = focus_pairwise(key, f"pairwise_oof_correlation_focus_{short_name}.csv")

# Backward-compatible variables for easy notebook inspection.
focus_040_df = focus_pairwise_tables.get("040", pd.DataFrame())
focus_039_df = focus_pairwise_tables.get("039", pd.DataFrame())
focus_038_df = focus_pairwise_tables.get("038", pd.DataFrame())
focus_037_df = focus_pairwise_tables.get("037", pd.DataFrame())
focus_033_df = focus_pairwise_tables.get("033", pd.DataFrame())


,key,exp_id,family,role,cv_from_memo,public_lb,public_lb_biased,tuned_cv,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
0,040_gpu_logreg_add039,exp_20260611_040_gpu_logreg_stacker_add039_own,GPU_LogisticRegression_Stacker,current_best_after_add039,0.970202,0.970690,NaN,NaN,0.970202,0.0,0.959500,0.976866,0.974240
1,038_gpu_logreg_add037,exp_20260610_038_gpu_logreg_stacker_add037_own,GPU_LogisticRegression_Stacker,previous_stacker_best_before_040,0.970135,0.970600,NaN,NaN,0.970135,0.0,0.958120,0.977950,0.974336
2,036_blend_add035,exp_20260610_036_blend_add035_shared001_check,Blend,cv_best_but_public_weak_add035_blend,0.970073,0.970230,NaN,NaN,0.970073,0.0,0.960777,0.976943,0.972499
3,036_gpu_logreg_add035,exp_20260610_036_gpu_logreg_stacker_add035_own,GPU_LogisticRegression_Stacker,improved_stacker_backup_add035,0.970067,0.970370,NaN,NaN,0.970067,0.0,0.958313,0.978701,0.973188
4,033_blend_add032,exp_20260609_033_blend_add032_tabm_check,Blend,previous_public_confirmed_best_backup,0.970040,0.970430,NaN,NaN,0.970040,0.0,0.961775,0.975773,0.972571
5,034_gpu_logreg_default,exp_20260609_034_gpu_logreg_stacker_own,GPU_LogisticRegression_Stacker,stacker_backup_default,0.970037,0.970220,NaN,NaN,0.970037,0.0,0.959794,0.977771,0.972547
6,031_blend_add030,exp_20260608_031_blend_add030_ovr_xgb_check,Blend,public_confirmed_current_best_before_033,0.970033,0.970430,NaN,NaN,0.970033,0.0,0.961738,0.975790,0.972571
7,029_blend_add028,exp_20260608_029_blend_add028_cat_v3_check,Blend,previous_best_backup,0.970002,0.970036,NaN,NaN,0.970002,0.0,0.961315,0.975867,0.972825
8,027_blend_add026,exp_20260608_027_blend_add026_realmlp_v5_check,Blend,previous_cv_trusted_slot,0.969543,0.969750,NaN,NaN,0.969543,0.0,0.961479,0.976439,0.970710
9,024_seed_ensemble_blend,exp_20260608_024_seed_ensemble_and_blend_check,Blend,seed_ensemble_blend_reference,0.969480,0.969580,NaN,NaN,0.969480,0.0,0.961956,0.976174,0.970311


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,030_ovr_xgb,037_tabicl_v2,0.974477,0.953569,0.037044,0.028955,0.049665,0.020975,0.363871,0.007980,0.028690
1,032_ovr_tabm,037_tabicl_v2,0.974850,0.953750,0.037186,0.029062,0.049665,0.020963,0.362909,0.008099,0.028702
2,015_realmlp_seed42,037_tabicl_v2,0.979561,0.910951,0.031806,0.034388,0.049665,0.026350,0.456641,0.008038,0.023315
3,017_realmlp_bias,037_tabicl_v2,0.980194,0.901659,0.031480,0.035542,0.049665,0.027077,0.465809,0.008465,0.022588
4,035_shared001_updated,037_tabicl_v2,0.981919,0.878275,0.029999,0.034984,0.049665,0.027516,0.481598,0.007469,0.022150
...,...,...,...,...,...,...,...,...,...,...,...
226,036_gpu_logreg_add035,038_gpu_logreg_add037,0.999909,0.998891,0.001909,0.035419,0.035533,0.034539,0.948533,0.000880,0.000994
227,034_gpu_logreg_default,036_gpu_logreg_add035,0.999938,0.998882,0.001436,0.034731,0.035419,0.034366,0.960358,0.000365,0.001053
228,029_blend_add028,033_blend_add032,0.999993,0.999794,0.000438,0.034083,0.033838,0.033749,0.987632,0.000334,0.000088
229,029_blend_add028,031_blend_add030,0.999993,0.999801,0.000421,0.034083,0.033858,0.033768,0.988140,0.000315,0.000090


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,040_gpu_logreg_add039,0.984338,0.953540,0.027484,0.049665,0.034865,0.028666,0.513131,0.021000,0.006199
1,030_ovr_xgb,040_gpu_logreg_add039,0.989809,0.940546,0.021798,0.028955,0.034865,0.021114,0.494403,0.007841,0.013751
2,032_ovr_tabm,040_gpu_logreg_add039,0.990036,0.949985,0.021753,0.029062,0.034865,0.021195,0.496007,0.007867,0.013669
3,018_xgb_shared003,040_gpu_logreg_add039,0.994166,0.963306,0.014267,0.033536,0.034865,0.027134,0.657545,0.006402,0.007730
4,015_realmlp_seed42,040_gpu_logreg_add039,0.996843,0.927797,0.010510,0.034388,0.034865,0.029495,0.741875,0.004893,0.005369
5,017_realmlp_bias,040_gpu_logreg_add039,0.996997,0.925200,0.010458,0.035542,0.034865,0.030093,0.746466,0.005449,0.004772
6,026_realmlp_v5,040_gpu_logreg_add039,0.997798,0.861649,0.008385,0.035388,0.034865,0.031044,0.791757,0.004344,0.003821
7,035_shared001_updated,040_gpu_logreg_add039,0.997811,0.902374,0.008593,0.034984,0.034865,0.030728,0.785487,0.004256,0.004136
8,039_cat_v3_seed2026_qbin_variant,040_gpu_logreg_add039,0.998130,0.968624,0.008123,0.035587,0.034865,0.031245,0.796916,0.004342,0.003620
9,019_blend_best,040_gpu_logreg_add039,0.998178,0.940981,0.008153,0.032528,0.034865,0.029693,0.787605,0.002835,0.005172


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,039_cat_v3_seed2026_qbin_variant,0.984865,0.961404,0.027367,0.049665,0.035587,0.029088,0.517918,0.020577,0.006499
1,030_ovr_xgb,039_cat_v3_seed2026_qbin_variant,0.988783,0.964342,0.022205,0.028955,0.035587,0.021308,0.492849,0.007647,0.014279
2,032_ovr_tabm,039_cat_v3_seed2026_qbin_variant,0.988993,0.962610,0.022210,0.029062,0.035587,0.021362,0.493478,0.007701,0.014225
3,018_xgb_shared003,039_cat_v3_seed2026_qbin_variant,0.992109,0.967151,0.016841,0.033536,0.035587,0.026253,0.612379,0.007283,0.009334
4,015_realmlp_seed42,039_cat_v3_seed2026_qbin_variant,0.993330,0.916658,0.015687,0.034388,0.035587,0.027315,0.640276,0.007074,0.008272
5,017_realmlp_bias,039_cat_v3_seed2026_qbin_variant,0.993395,0.907284,0.015802,0.035542,0.035587,0.027822,0.642443,0.007720,0.007765
6,026_realmlp_v5,039_cat_v3_seed2026_qbin_variant,0.994581,0.804236,0.014011,0.035388,0.035587,0.028631,0.676157,0.006757,0.006956
7,035_shared001_updated,039_cat_v3_seed2026_qbin_variant,0.994848,0.898905,0.013708,0.034984,0.035587,0.028567,0.680096,0.006417,0.007020
8,019_blend_best,039_cat_v3_seed2026_qbin_variant,0.995390,0.929347,0.013067,0.032528,0.035587,0.027647,0.683188,0.004881,0.007940
9,020_blend_bias,039_cat_v3_seed2026_qbin_variant,0.995649,0.926949,0.012982,0.034489,0.035587,0.028667,0.692308,0.005821,0.006920


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,037_tabicl_v2,038_gpu_logreg_add037,0.984509,0.953093,0.027354,0.049665,0.035533,0.029073,0.517992,0.020592,0.006461
1,030_ovr_xgb,038_gpu_logreg_add037,0.989016,0.936473,0.022898,0.028955,0.035533,0.020903,0.479574,0.008052,0.014631
2,032_ovr_tabm,038_gpu_logreg_add037,0.989262,0.945291,0.022799,0.029062,0.035533,0.021006,0.481920,0.008056,0.014527
3,018_xgb_shared003,038_gpu_logreg_add037,0.993836,0.961689,0.014768,0.033536,0.035533,0.027219,0.650401,0.006317,0.008314
4,015_realmlp_seed42,038_gpu_logreg_add037,0.996779,0.922695,0.010654,0.034388,0.035533,0.029757,0.740869,0.004632,0.005776
5,017_realmlp_bias,038_gpu_logreg_add037,0.996982,0.920486,0.010491,0.035542,0.035533,0.030408,0.747732,0.005134,0.005125
6,038_gpu_logreg_add037,039_cat_v3_seed2026_qbin_variant,0.997587,0.963981,0.009395,0.035533,0.035587,0.030954,0.770634,0.004580,0.004633
7,035_shared001_updated,038_gpu_logreg_add037,0.997770,0.899700,0.008749,0.034984,0.035533,0.030988,0.783937,0.003996,0.004545
8,026_realmlp_v5,038_gpu_logreg_add037,0.997836,0.869126,0.008323,0.035388,0.035533,0.031407,0.794854,0.003980,0.004126
9,019_blend_best,038_gpu_logreg_add037,0.997989,0.935244,0.008709,0.032528,0.035533,0.029750,0.776527,0.002778,0.005783


...

,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,036_gpu_logreg_add035,037_tabicl_v2,0.985368,0.955626,0.026473,0.035419,0.049665,0.029461,0.529644,0.005958,0.020204
1,030_ovr_xgb,036_gpu_logreg_add035,0.989202,0.937636,0.022671,0.028955,0.035419,0.020958,0.482726,0.007997,0.014461
2,032_ovr_tabm,036_gpu_logreg_add035,0.989481,0.946766,0.022555,0.029062,0.035419,0.021069,0.485318,0.007993,0.014350
3,018_xgb_shared003,036_gpu_logreg_add035,0.993900,0.961192,0.014658,0.033536,0.035419,0.027223,0.652320,0.006313,0.008196
4,015_realmlp_seed42,036_gpu_logreg_add035,0.996811,0.924964,0.010680,0.034388,0.035419,0.029677,0.739523,0.004711,0.005742
5,017_realmlp_bias,036_gpu_logreg_add035,0.996961,0.922292,0.010595,0.035542,0.035419,0.030294,0.744921,0.005248,0.005125
6,036_gpu_logreg_add035,039_cat_v3_seed2026_qbin_variant,0.997605,0.963800,0.009374,0.035419,0.035587,0.030907,0.770766,0.004512,0.004680
7,035_shared001_updated,036_gpu_logreg_add035,0.997797,0.900401,0.008695,0.034984,0.035419,0.030954,0.784642,0.004031,0.004465
8,026_realmlp_v5,036_gpu_logreg_add035,0.997834,0.867962,0.008359,0.035388,0.035419,0.031328,0.793533,0.004060,0.004091
9,019_blend_best,036_gpu_logreg_add035,0.998015,0.937168,0.008537,0.032528,0.035419,0.029776,0.780062,0.002752,0.005643


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,036_blend_add035,037_tabicl_v2,0.985093,0.904353,0.027160,0.034264,0.049665,0.028541,0.515292,0.005723,0.021124
1,030_ovr_xgb,036_blend_add035,0.990463,0.869145,0.020823,0.028955,0.034264,0.021310,0.508472,0.007645,0.012954
2,032_ovr_tabm,036_blend_add035,0.990711,0.879875,0.020667,0.029062,0.034264,0.021429,0.511472,0.007633,0.012835
3,018_xgb_shared003,036_blend_add035,0.994216,0.903492,0.014182,0.033536,0.034264,0.026889,0.657240,0.006648,0.007375
4,015_realmlp_seed42,036_blend_add035,0.997365,0.899824,0.009616,0.034388,0.034264,0.029623,0.759020,0.004765,0.004640
5,017_realmlp_bias,036_blend_add035,0.997437,0.908192,0.009705,0.035542,0.034264,0.030150,0.760297,0.005392,0.004114
6,036_blend_add035,039_cat_v3_seed2026_qbin_variant,0.997845,0.912962,0.008929,0.034264,0.035587,0.030550,0.777347,0.003714,0.005037
7,035_shared001_updated,036_blend_add035,0.998337,0.880209,0.007484,0.034984,0.034264,0.030968,0.808968,0.004017,0.003296
8,026_realmlp_v5,036_blend_add035,0.998402,0.946615,0.007200,0.035388,0.034264,0.031309,0.816551,0.004079,0.002955
9,028_cat_v3,036_blend_add035,0.998477,0.915623,0.007612,0.035277,0.034264,0.031045,0.806479,0.004231,0.003218


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,033_blend_add032,037_tabicl_v2,0.984870,0.865547,0.027770,0.033838,0.049665,0.028016,0.504916,0.005821,0.021649
1,030_ovr_xgb,033_blend_add032,0.990558,0.821940,0.020357,0.028955,0.033838,0.021344,0.514960,0.007611,0.012493
2,032_ovr_tabm,033_blend_add032,0.990801,0.830690,0.020230,0.029062,0.033838,0.021446,0.517361,0.007616,0.012391
3,018_xgb_shared003,033_blend_add032,0.993883,0.858653,0.014542,0.033536,0.033838,0.026513,0.648849,0.007024,0.007325
4,015_realmlp_seed42,033_blend_add032,0.997205,0.868130,0.009731,0.034388,0.033838,0.029355,0.755191,0.005033,0.004483
5,017_realmlp_bias,033_blend_add032,0.997258,0.881385,0.009920,0.035542,0.033838,0.029828,0.754149,0.005714,0.004010
6,033_blend_add032,039_cat_v3_seed2026_qbin_variant,0.997930,0.870715,0.008723,0.033838,0.035587,0.030439,0.780789,0.003398,0.005148
7,033_blend_add032,035_shared001_updated,0.998127,0.842031,0.007886,0.033838,0.034984,0.030554,0.798407,0.003284,0.004431
8,019_blend_best,033_blend_add032,0.998300,0.878981,0.007554,0.032528,0.033838,0.029478,0.799127,0.003050,0.004360
9,020_blend_bias,033_blend_add032,0.998542,0.899930,0.007273,0.034489,0.033838,0.030592,0.810704,0.003897,0.003246


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,031_blend_add030,037_tabicl_v2,0.984873,0.865479,0.027770,0.033858,0.049665,0.028026,0.505009,0.005832,0.021639
1,030_ovr_xgb,031_blend_add030,0.990536,0.821836,0.020385,0.028955,0.033858,0.021341,0.514576,0.007614,0.012518
2,031_blend_add030,032_ovr_tabm,0.990777,0.830573,0.020262,0.033858,0.029062,0.021441,0.516912,0.012417,0.007621
3,018_xgb_shared003,031_blend_add030,0.993874,0.858577,0.014553,0.033536,0.033858,0.026518,0.648729,0.007018,0.007340
4,015_realmlp_seed42,031_blend_add030,0.997199,0.868061,0.009734,0.034388,0.033858,0.029364,0.755178,0.005025,0.004495
5,017_realmlp_bias,031_blend_add030,0.997253,0.881326,0.009913,0.035542,0.033858,0.029842,0.754368,0.005700,0.004017
6,031_blend_add030,039_cat_v3_seed2026_qbin_variant,0.997933,0.870672,0.008719,0.033858,0.035587,0.030451,0.780927,0.003407,0.005136
7,031_blend_add030,035_shared001_updated,0.998123,0.841979,0.007883,0.033858,0.034984,0.030566,0.798543,0.003293,0.004418
8,019_blend_best,031_blend_add030,0.998293,0.878896,0.007571,0.032528,0.033858,0.029480,0.798761,0.003048,0.004379
9,020_blend_bias,031_blend_add030,0.998538,0.899859,0.007276,0.034489,0.033858,0.030600,0.810673,0.003888,0.003258


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,028_cat_v3,037_tabicl_v2,0.984692,0.962312,0.027529,0.035277,0.049665,0.028853,0.514406,0.006424,0.020812
1,028_cat_v3,030_ovr_xgb,0.988941,0.965161,0.021862,0.035277,0.028955,0.021330,0.497194,0.013947,0.007625
2,028_cat_v3,032_ovr_tabm,0.989126,0.963762,0.021803,0.035277,0.029062,0.021407,0.498608,0.013870,0.007656
3,018_xgb_shared003,028_cat_v3,0.992356,0.969802,0.016441,0.033536,0.035277,0.026291,0.618289,0.007245,0.008986
4,015_realmlp_seed42,028_cat_v3,0.993619,0.913554,0.015279,0.034388,0.035277,0.027358,0.646647,0.007030,0.007919
5,017_realmlp_bias,028_cat_v3,0.993673,0.904245,0.015462,0.035542,0.035277,0.027832,0.647474,0.007709,0.007444
6,026_realmlp_v5,028_cat_v3,0.994791,0.804927,0.013787,0.035388,0.035277,0.028589,0.679483,0.006798,0.006687
7,028_cat_v3,035_shared001_updated,0.995089,0.900310,0.013491,0.035277,0.034984,0.028527,0.683544,0.006750,0.006457
8,019_blend_best,028_cat_v3,0.995654,0.926925,0.012703,0.032528,0.035277,0.027673,0.689555,0.004855,0.007604
9,024_seed_ensemble_blend,028_cat_v3,0.995894,0.936706,0.012668,0.033962,0.035277,0.028413,0.695940,0.005550,0.006864


In [5]:
# ============================================================
# 4. Blend diagnostics
# ============================================================

blend_rows = {}
blend_probas = {}

def _weight_of(keys, weights, target_key):
    if weights is None or target_key not in keys:
        return None
    return float(dict(zip(keys, weights)).get(target_key, 0.0))

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    for short_name, target_key in TRACK_KEYS.items():
        row[f"includes_{short_name}"] = target_key in keys
        row[f"weight_{short_name}"] = _weight_of(keys, weights, target_key)
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row
    if test_p is not None:
        blend_probas[(set_name, method)] = (normalize_rows(oof_p), normalize_rows(test_p))

for set_name, keys in BLEND_SETS.items():
    print("\n===", set_name, keys, "===")

    # 1) Simple average.
    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred)

    # 2) Hill climb nonnegative raw probabilities.
    try:
        w, score, hist = hill_climb_weights(y, [oof[k] for k in keys])
        hc_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        hc_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=w, extra={"hc_score_internal": score, "hc_n_steps": len(hist)})
    except Exception as e:
        print(f"[WARN] HC failed: {set_name}: {e}")

    # 3) Nelder-Mead softmax weights.
    try:
        w, score, res = nm_softmax_weights(y, [oof[k] for k in keys])
        nm_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        nm_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=w, extra={"nm_score_internal": score, "nm_success": bool(res.success), "nm_message": str(res.message)})
    except Exception as e:
        print(f"[WARN] NM failed: {set_name}: {e}")

    # 4) In-sample meta models. Diagnostic only; do not use as final candidates.
    for mode in ["raw", "rank", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, multi_class="multinomial", solver="lbfgs", max_iter=2000, random_state=SEED)
            lr.fit(X_meta, y)
            record_blend(set_name, f"logreg_{mode}", keys, lr.predict_proba(X_meta), lr.predict_proba(X_test_meta), extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            record_blend(set_name, f"ridgeclf_{mode}", keys, softmax_np(rc.decision_function(X_meta)), softmax_np(rc.decision_function(X_test_meta)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            record_blend(set_name, f"ridge_reg_{mode}", keys, normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None)), normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            oof_cols, test_cols = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                oof_cols.append(en.predict(X_meta))
                test_cols.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(oof_cols).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(test_cols).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(blend_df.head(200))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(safe_blend_df.head(100))
safe_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics.csv", index=False)

focus_blend_tables = {}
for short_name, target_key in TRACK_KEYS.items():
    include_col = f"includes_{short_name}"
    if include_col in safe_blend_df.columns:
        df = safe_blend_df[safe_blend_df[include_col]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
        focus_blend_tables[short_name] = df
        display(df.head(80))
        df.to_csv(OUTDIR / f"safe_blend_diagnostics_focus_{short_name}.csv", index=False)

# Backward-compatible variables for easy notebook inspection.
focus_040_blend_df = focus_blend_tables.get("040", pd.DataFrame())
focus_039_blend_df = focus_blend_tables.get("039", pd.DataFrame())
focus_038_blend_df = focus_blend_tables.get("038", pd.DataFrame())
focus_037_blend_df = focus_blend_tables.get("037", pd.DataFrame())
focus_033_blend_df = focus_blend_tables.get("033", pd.DataFrame())



=== A_040_only ['040_gpu_logreg_add039'] ===

=== B_038_only ['038_gpu_logreg_add037'] ===

=== C_033_only ['033_blend_add032'] ===

=== D_036_gpu_only ['036_gpu_logreg_add035'] ===

=== E_036_blend_only ['036_blend_add035'] ===

=== F_039_only ['039_cat_v3_seed2026_qbin_variant'] ===

=== G_028_only ['028_cat_v3'] ===

=== H_037_only ['037_tabicl_v2'] ===

=== I_031_only ['031_blend_add030'] ===

=== J_040_033 ['040_gpu_logreg_add039', '033_blend_add032'] ===

=== K_040_038 ['040_gpu_logreg_add039', '038_gpu_logreg_add037'] ===

=== L_040_039 ['040_gpu_logreg_add039', '039_cat_v3_seed2026_qbin_variant'] ===

=== M_040_028 ['040_gpu_logreg_add039', '028_cat_v3'] ===

=== N_040_036gpu ['040_gpu_logreg_add039', '036_gpu_logreg_add035'] ===

=== O_040_036blend ['040_gpu_logreg_add039', '036_blend_add035'] ===

=== P_040_037 ['040_gpu_logreg_add039', '037_tabicl_v2'] ===

=== Q_033_039 ['033_blend_add032', '039_cat_v3_seed2026_qbin_variant'] ===

=== R_038_039 ['038_gpu_logreg_add037', '0

,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z1_040_033_038_036gpu_036blend_039_037,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",7,0.970237,"{""040_gpu_logreg_add039"": 0.30521971714214097,...",True,0.305220,True,0.000000,...,0.973623,0.970237,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,X_040_033_036gpu_036blend_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,036_gpu...",5,0.970233,"{""040_gpu_logreg_add039"": 0.4647208501420461, ...",True,0.464721,True,0.000000,...,0.973635,0.970233,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T_040_033_039,nm_softmax_raw,"040_gpu_logreg_add039,033_blend_add032,039_cat...",3,0.970208,"{""040_gpu_logreg_add039"": 0.9949517141931713, ...",True,0.994952,True,0.000524,...,0.974240,NaN,NaN,0.970208,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,M_040_028,nm_softmax_raw,"040_gpu_logreg_add039,028_cat_v3",2,0.970205,"{""040_gpu_logreg_add039"": 0.9989348101432777, ...",True,0.998935,False,NaN,...,0.974240,NaN,NaN,0.970205,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,M_040_028,hc_nonnegative_raw,"040_gpu_logreg_add039,028_cat_v3",2,0.970205,"{""040_gpu_logreg_add039"": 0.998003992015968, ""...",True,0.998004,False,NaN,...,0.974228,0.970205,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,C_033_only,ridge_reg_raw_rank_logit,033_blend_add032,1,0.968780,None,False,NaN,False,NaN,...,0.963106,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
196,Q_033_039,elasticnet_reg_raw_logit,"033_blend_add032,039_cat_v3_seed2026_qbin_variant",2,0.968749,None,False,NaN,True,NaN,...,0.963686,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,0.0005,0.05
197,J_040_033,elasticnet_reg_raw_logit,"040_gpu_logreg_add039,033_blend_add032",2,0.968734,None,True,NaN,False,NaN,...,0.961909,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,0.0005,0.05
198,L_040_039,ridgeclf_raw_logit,"040_gpu_logreg_add039,039_cat_v3_seed2026_qbin...",2,0.968714,None,True,NaN,True,NaN,...,0.961740,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z1_040_033_038_036gpu_036blend_039_037,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",7,0.970237,"{""040_gpu_logreg_add039"": 0.30521971714214097,...",True,0.305220,True,0.000000,...,0.973623,0.970237,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,X_040_033_036gpu_036blend_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,036_gpu...",5,0.970233,"{""040_gpu_logreg_add039"": 0.4647208501420461, ...",True,0.464721,True,0.000000,...,0.973635,0.970233,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T_040_033_039,nm_softmax_raw,"040_gpu_logreg_add039,033_blend_add032,039_cat...",3,0.970208,"{""040_gpu_logreg_add039"": 0.9949517141931713, ...",True,0.994952,True,0.000524,...,0.974240,NaN,NaN,0.970208,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,M_040_028,nm_softmax_raw,"040_gpu_logreg_add039,028_cat_v3",2,0.970205,"{""040_gpu_logreg_add039"": 0.9989348101432777, ...",True,0.998935,False,NaN,...,0.974240,NaN,NaN,0.970205,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,M_040_028,hc_nonnegative_raw,"040_gpu_logreg_add039,028_cat_v3",2,0.970205,"{""040_gpu_logreg_add039"": 0.998003992015968, ""...",True,0.998004,False,NaN,...,0.974228,0.970205,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,F_039_only,avg,039_cat_v3_seed2026_qbin_variant,1,0.968705,None,False,NaN,True,NaN,...,0.971616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,P_040_037,avg,"040_gpu_logreg_add039,037_tabicl_v2",2,0.967451,None,True,NaN,False,NaN,...,0.973285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,H_037_only,nm_softmax_raw,037_tabicl_v2,1,0.958077,"{""037_tabicl_v2"": 1.0}",False,NaN,False,NaN,...,0.961970,NaN,NaN,0.958077,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
85,H_037_only,avg,037_tabicl_v2,1,0.958077,None,False,NaN,False,NaN,...,0.961970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z1_040_033_038_036gpu_036blend_039_037,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",7,0.970237,"{""040_gpu_logreg_add039"": 0.30521971714214097,...",True,0.305220,True,0.000000,...,0.973623,0.970237,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,X_040_033_036gpu_036blend_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,036_gpu...",5,0.970233,"{""040_gpu_logreg_add039"": 0.4647208501420461, ...",True,0.464721,True,0.000000,...,0.973635,0.970233,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T_040_033_039,nm_softmax_raw,"040_gpu_logreg_add039,033_blend_add032,039_cat...",3,0.970208,"{""040_gpu_logreg_add039"": 0.9949517141931713, ...",True,0.994952,True,0.000524,...,0.974240,NaN,NaN,0.970208,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,M_040_028,nm_softmax_raw,"040_gpu_logreg_add039,028_cat_v3",2,0.970205,"{""040_gpu_logreg_add039"": 0.9989348101432777, ...",True,0.998935,False,NaN,...,0.974240,NaN,NaN,0.970205,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,M_040_028,hc_nonnegative_raw,"040_gpu_logreg_add039,028_cat_v3",2,0.970205,"{""040_gpu_logreg_add039"": 0.998003992015968, ""...",True,0.998004,False,NaN,...,0.974228,0.970205,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,P_040_037,nm_softmax_raw,"040_gpu_logreg_add039,037_tabicl_v2",2,0.970202,"{""040_gpu_logreg_add039"": 0.9998580997602273, ...",True,0.999858,False,NaN,...,0.974240,NaN,NaN,0.970202,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,L_040_039,hc_nonnegative_raw,"040_gpu_logreg_add039,039_cat_v3_seed2026_qbin...",2,0.970202,"{""040_gpu_logreg_add039"": 1.0, ""039_cat_v3_see...",True,1.000000,True,0.000000,...,0.974240,0.970202,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,A_040_only,avg,040_gpu_logreg_add039,1,0.970202,None,True,NaN,False,NaN,...,0.974240,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,P_040_037,hc_nonnegative_raw,"040_gpu_logreg_add039,037_tabicl_v2",2,0.970202,"{""040_gpu_logreg_add039"": 1.0, ""037_tabicl_v2""...",True,1.000000,False,NaN,...,0.974240,0.970202,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,A_040_only,hc_nonnegative_raw,040_gpu_logreg_add039,1,0.970202,"{""040_gpu_logreg_add039"": 1.0}",True,1.000000,False,NaN,...,0.974240,0.970202,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z1_040_033_038_036gpu_036blend_039_037,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",7,0.970237,"{""040_gpu_logreg_add039"": 0.30521971714214097,...",True,0.305220,True,0.000000,...,0.973623,0.970237,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,X_040_033_036gpu_036blend_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,036_gpu...",5,0.970233,"{""040_gpu_logreg_add039"": 0.4647208501420461, ...",True,0.464721,True,0.000000,...,0.973635,0.970233,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,T_040_033_039,nm_softmax_raw,"040_gpu_logreg_add039,033_blend_add032,039_cat...",3,0.970208,"{""040_gpu_logreg_add039"": 0.9949517141931713, ...",True,0.994952,True,0.000524,...,0.974240,NaN,NaN,0.970208,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,L_040_039,hc_nonnegative_raw,"040_gpu_logreg_add039,039_cat_v3_seed2026_qbin...",2,0.970202,"{""040_gpu_logreg_add039"": 1.0, ""039_cat_v3_see...",True,1.000000,True,0.000000,...,0.974240,0.970202,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.970190,"{""015_realmlp_seed42"": 0.037770043117670606, ""...",True,0.201390,True,0.139486,...,0.973188,0.970190,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,L_040_039,nm_softmax_raw,"040_gpu_logreg_add039,039_cat_v3_seed2026_qbin...",2,0.970190,"{""040_gpu_logreg_add039"": 0.9386639297681819, ...",True,0.938664,True,0.061336,...,0.974155,NaN,NaN,0.970190,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,T_040_033_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,039_cat...",3,0.970161,"{""040_gpu_logreg_add039"": 0.5995194233079695, ...",True,0.599519,True,0.000000,...,0.973539,0.970161,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Z3_core_no_040_033_038_036gpu_039,hc_nonnegative_raw,"033_blend_add032,038_gpu_logreg_add037,036_gpu...",4,0.970161,"{""033_blend_add032"": 0.29562518770351764, ""038...",False,NaN,True,0.019511,...,0.973454,0.970161,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,V_040_038_039,hc_nonnegative_raw,"040_gpu_logreg_add039,038_gpu_logreg_add037,03...",3,0.970159,"{""040_gpu_logreg_add039"": 0.3939393939393939, ...",True,0.393939,True,0.000000,...,0.974264,0.970159,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Z2_040_033_038_034_031_029_028_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.970146,"{""040_gpu_logreg_add039"": 0.23178034873941306,...",True,0.231780,True,0.039380,...,0.973176,0.970146,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


...

,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.970190,"{""015_realmlp_seed42"": 0.037770043117670606, ""...",True,0.201390,True,0.139486,...,0.973188,0.970190,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Y_027_026_028_039_030_032_035_037,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",8,0.969951,"{""027_blend_add026"": 0.2026055475759779, ""026_...",False,NaN,True,0.179956,...,0.971435,0.969951,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969838,"{""015_realmlp_seed42"": 0.04504467859324805, ""0...",True,0.053347,True,0.052263,...,0.970396,NaN,NaN,0.969838,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,Z_full_015_017_018_019_020_024_026_027_028_029...,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969801,None,True,NaN,True,NaN,...,0.970142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Y_027_026_028_039_030_032_035_037,nm_softmax_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",8,0.969534,"{""027_blend_add026"": 0.1410882842897569, ""026_...",False,NaN,True,0.167987,...,0.969283,NaN,NaN,0.969534,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,Y_027_026_028_039_030_032_035_037,avg,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",8,0.969369,None,False,NaN,True,NaN,...,0.967531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.970190,"{""015_realmlp_seed42"": 0.037770043117670606, ""...",True,0.201390,True,0.139486,...,0.973188,0.970190,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z2_040_033_038_034_031_029_028_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.970146,"{""040_gpu_logreg_add039"": 0.23178034873941306,...",True,0.231780,True,0.039380,...,0.973176,0.970146,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I_031_only,avg,031_blend_add030,1,0.970033,None,False,NaN,False,NaN,...,0.972571,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,I_031_only,nm_softmax_raw,031_blend_add030,1,0.970033,"{""031_blend_add030"": 1.0}",False,NaN,False,NaN,...,0.972571,NaN,NaN,0.970033,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
4,I_031_only,hc_nonnegative_raw,031_blend_add030,1,0.970033,"{""031_blend_add030"": 1.0}",False,NaN,False,NaN,...,0.972571,0.970033,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Z2_040_033_038_034_031_029_028_039,nm_softmax_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.970001,"{""040_gpu_logreg_add039"": 0.14179618436704058,...",True,0.141796,True,0.120928,...,0.973079,NaN,NaN,0.970001,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,Z2_040_033_038_034_031_029_028_039,avg,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.969968,None,True,NaN,True,NaN,...,0.972970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969838,"{""015_realmlp_seed42"": 0.04504467859324805, ""0...",True,0.053347,True,0.052263,...,0.970396,NaN,NaN,0.969838,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
8,Z_full_015_017_018_019_020_024_026_027_028_029...,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969801,None,True,NaN,True,NaN,...,0.970142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.970190,"{""015_realmlp_seed42"": 0.037770043117670606, ""...",True,0.201390,True,0.139486,...,0.973188,0.970190,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Y_027_026_028_039_030_032_035_037,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",8,0.969951,"{""027_blend_add026"": 0.2026055475759779, ""026_...",False,NaN,True,0.179956,...,0.971435,0.969951,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969838,"{""015_realmlp_seed42"": 0.04504467859324805, ""0...",True,0.053347,True,0.052263,...,0.970396,NaN,NaN,0.969838,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,Z_full_015_017_018_019_020_024_026_027_028_029...,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969801,None,True,NaN,True,NaN,...,0.970142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Y_027_026_028_039_030_032_035_037,nm_softmax_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",8,0.969534,"{""027_blend_add026"": 0.1410882842897569, ""026_...",False,NaN,True,0.167987,...,0.969283,NaN,NaN,0.969534,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,Y_027_026_028_039_030_032_035_037,avg,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",8,0.969369,None,False,NaN,True,NaN,...,0.967531,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.970190,"{""015_realmlp_seed42"": 0.037770043117670606, ""...",True,0.201390,True,0.139486,...,0.973188,0.970190,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Z2_040_033_038_034_031_029_028_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.970146,"{""040_gpu_logreg_add039"": 0.23178034873941306,...",True,0.231780,True,0.039380,...,0.973176,0.970146,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z2_040_033_038_034_031_029_028_039,nm_softmax_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.970001,"{""040_gpu_logreg_add039"": 0.14179618436704058,...",True,0.141796,True,0.120928,...,0.973079,NaN,NaN,0.970001,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
3,Z2_040_033_038_034_031_029_028_039,avg,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.969968,None,True,NaN,True,NaN,...,0.972970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969838,"{""015_realmlp_seed42"": 0.04504467859324805, ""0...",True,0.053347,True,0.052263,...,0.970396,NaN,NaN,0.969838,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,Z_full_015_017_018_019_020_024_026_027_028_029...,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969801,None,True,NaN,True,NaN,...,0.970142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,weights_json,includes_040,weight_040,includes_039,weight_039,...,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,M_040_028,nm_softmax_raw,"040_gpu_logreg_add039,028_cat_v3",2,0.970205,"{""040_gpu_logreg_add039"": 0.9989348101432777, ...",True,0.998935,False,NaN,...,0.974240,NaN,NaN,0.970205,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
1,M_040_028,hc_nonnegative_raw,"040_gpu_logreg_add039,028_cat_v3",2,0.970205,"{""040_gpu_logreg_add039"": 0.998003992015968, ""...",True,0.998004,False,NaN,...,0.974228,0.970205,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Z_full_015_017_018_019_020_024_026_027_028_029...,hc_nonnegative_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.970190,"{""015_realmlp_seed42"": 0.037770043117670606, ""...",True,0.201390,True,0.139486,...,0.973188,0.970190,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Z2_040_033_038_034_031_029_028_039,hc_nonnegative_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.970146,"{""040_gpu_logreg_add039"": 0.23178034873941306,...",True,0.231780,True,0.039380,...,0.973176,0.970146,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Z2_040_033_038_034_031_029_028_039,nm_softmax_raw,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.970001,"{""040_gpu_logreg_add039"": 0.14179618436704058,...",True,0.141796,True,0.120928,...,0.973079,NaN,NaN,0.970001,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
5,Z2_040_033_038_034_031_029_028_039,avg,"040_gpu_logreg_add039,033_blend_add032,038_gpu...",8,0.969968,None,True,NaN,True,NaN,...,0.972970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Y_027_026_028_039_030_032_035_037,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3,039...",8,0.969951,"{""027_blend_add026"": 0.2026055475759779, ""026_...",False,NaN,True,0.179956,...,0.971435,0.969951,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Z_full_015_017_018_019_020_024_026_027_028_029...,nm_softmax_raw,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969838,"{""015_realmlp_seed42"": 0.04504467859324805, ""0...",True,0.053347,True,0.052263,...,0.970396,NaN,NaN,0.969838,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
8,Z_full_015_017_018_019_020_024_026_027_028_029...,avg,"015_realmlp_seed42,017_realmlp_bias,018_xgb_sh...",22,0.969801,None,True,NaN,True,NaN,...,0.970142,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,M_040_028,avg,"040_gpu_logreg_add039,028_cat_v3",2,0.969718,None,True,NaN,False,NaN,...,0.973055,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# 5. Save best safe blend
# ============================================================

safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
else:
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * oof[best_keys[i]] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred[best_keys[i]] for i in range(len(best_keys))))

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / f"oof_{EXP_ID}_best_blend_proba.npy"
best_pred_npy = OUTDIR / f"pred_{EXP_ID}_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / f"submission_{EXP_ID}_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
for short_name, target_key in TRACK_KEYS.items():
    best_info[f"includes_{short_name}"] = target_key in best_keys
    best_info[f"weight_{short_name}"] = best_row.get(f"weight_{short_name}")

save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_record = {
    "set_name": best_set_name,
    "method": best_method,
    "cv": float(best_oof_score),
    "keys": ",".join(best_keys),
    "weights_json": best_row.get("weights_json"),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}
for short_name, target_key in TRACK_KEYS.items():
    saved_submission_record[f"includes_{short_name}"] = target_key in best_keys
    saved_submission_record[f"weight_{short_name}"] = best_row.get(f"weight_{short_name}")

saved_submission_df = pd.DataFrame([saved_submission_record])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)


Best safe blend:
{
  "set_name": "Z1_040_033_038_036gpu_036blend_039_037",
  "method": "hc_nonnegative_raw",
  "keys": "040_gpu_logreg_add039,033_blend_add032,038_gpu_logreg_add037,036_gpu_logreg_add035,036_blend_add035,039_cat_v3_seed2026_qbin_variant,037_tabicl_v2",
  "n_models": 7,
  "balanced_accuracy": 0.9702371689099357,
  "weights_json": "{\"040_gpu_logreg_add039\": 0.30521971714214097, \"033_blend_add032\": 0.17195477022092448, \"038_gpu_logreg_add037\": 0.1769199642110537, \"036_gpu_logreg_add035\": 0.17195477022092448, \"036_blend_add035\": 0.17395077820495644, \"039_cat_v3_seed2026_qbin_variant\": 0.0, \"037_tabicl_v2\": 0.0}",
  "includes_040": true,
  "weight_040": 0.30521971714214097,
  "includes_039": true,
  "weight_039": 0.0,
  "includes_038": true,
  "weight_038": 0.1769199642110537,
  "includes_037": true,
  "weight_037": 0.0,
  "includes_036_gpu": true,
  "weight_036_gpu": 0.17195477022092448,
  "includes_036_blend": true,
  "weight_036_blend": 0.17395077820495644,


,set_name,method,cv,keys,weights_json,submission,oof_proba,pred_proba,includes_040,weight_040,...,includes_032,weight_032,includes_031,weight_031,includes_030,weight_030,includes_029,weight_029,includes_028,weight_028
0,Z1_040_033_038_036gpu_036blend_039_037,hc_nonnegative_raw,0.970237,"040_gpu_logreg_add039,033_blend_add032,038_gpu...","{""040_gpu_logreg_add039"": 0.30521971714214097,...",submission_exp_20260611_041_blend_add039_gpu04...,oof_exp_20260611_041_blend_add039_gpu040_check...,pred_exp_20260611_041_blend_add039_gpu040_chec...,True,0.30522,...,False,NaN,False,NaN,False,NaN,False,NaN,False,NaN


In [7]:
# ============================================================
# 6. Save summary / memo
# ============================================================

reference_scores = {
    "040_gpu_logreg_add039_cv": 0.9702017877238539,
    "040_gpu_logreg_add039_public_lb": 0.97069,
    "038_gpu_logreg_add037_cv": 0.9701353365798572,
    "038_gpu_logreg_add037_public_lb": 0.97060,
    "033_cv": 0.9700400336552478,
    "033_public_lb": 0.97043,
    "036_gpu_logreg_add035_cv": 0.9700674063284508,
    "036_gpu_logreg_add035_public_lb": 0.97037,
    "036_blend_add035_cv": 0.9700727843277738,
    "036_blend_add035_public_lb": 0.97023,
    "039_cat_v3_variant_cv": 0.9687053023092482,
    "039_cat_v3_variant_public_lb": 0.96984,
    "028_cat_v3_cv": 0.9688146470512534,
    "028_cat_v3_public_lb": 0.96969,
    "037_cv": 0.9580770153777599,
    "037_public_lb": 0.95920,
}

best_focus_safe = {}
for short_name, df in focus_blend_tables.items():
    best_focus_safe[short_name] = None if len(df) == 0 else df.iloc[0].to_dict()

judgement = {
    "reference_scores": reference_scores,
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "best_focus_safe": best_focus_safe,
    "main_questions": {
        "does_039_help_probability_blend": "Check focus_039 safe blends and weight_039.",
        "can_040_be_improved_by_simple_blend": "Check focus_040 safe blends versus 040 singleton CV/Public LB.",
        "does_033_still_work_as_diversity_backup": "Check 040+033 and broader sets containing 033.",
        "should_promote_041_best_blend": "Only if best safe blend improves CV and Public LB over 040 without being only an in-sample meta artifact.",
    },
    "automatic_view": {
        "best_safe_includes_040": bool(best_info.get("includes_040")),
        "best_safe_weight_040": best_info.get("weight_040"),
        "best_safe_includes_039": bool(best_info.get("includes_039")),
        "best_safe_weight_039": best_info.get("weight_039"),
        "best_safe_includes_033": bool(best_info.get("includes_033")),
        "best_safe_weight_033": best_info.get("weight_033"),
        "best_safe_cv": float(best_info["cv_score"]),
        "040_reference_cv": reference_scores["040_gpu_logreg_add039_cv"],
        "040_reference_public_lb": reference_scores["040_gpu_logreg_add039_public_lb"],
        "promotion_rule": (
            "Promote 041 only if the best safe probability blend beats 040 on CV and also confirms on Public LB. "
            "Do not promote in-sample LogReg/Ridge/ElasticNet diagnostic rows."
        ),
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Blend/correlation check after adding 039 CatBoost variant and 040 current GPU LogReg stacker to the candidate pool",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus": {k: v.to_dict(orient="records") for k, v in focus_pairwise_tables.items()},
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus": {k: v.to_dict(orient="records") for k, v in focus_blend_tables.items()},
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
summary_path = OUTDIR / "blend_add039_gpu040_summary.json"
save_json(summary, summary_path)

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 039 CatBoost variant and 040 GPU LogReg stacker",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-11",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from npy-files, add 039 CatBoost variant and 040 GPU LogReg stacker, "
            "and decide whether a direct probability blend can improve over the current 040 best."
        ),
        "success_criteria": [
            "load existing OOF/pred npy files including 038/039/040",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations with focus tables for 040/039/038/033",
            "evaluate add039/add040 blend sets",
            "separate safe blends from in-sample meta screening",
            "save best safe blend OOF/pred npy",
            "save best safe blend submission",
        ],
    },
    "inputs": {"npy_dir": str(NPY_DIR), "models": resolved_specs, "blend_sets": BLEND_SETS},
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "pairwise_oof_correlation_focus_prefix": "pairwise_oof_correlation_focus_*.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "safe_blend_diagnostics": "safe_blend_diagnostics.csv",
        "safe_blend_diagnostics_focus_prefix": "safe_blend_diagnostics_focus_*.csv",
        "best_blend_info": "best_blend_info.json",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "summary": summary_path.name,
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "040 is current best. 041 should only replace it if safe probability blend improves CV and Public LB. "
            "039 may help as direct blend material or only through GPU LogReg stacker."
        ),
        "automatic_view": judgement["automatic_view"],
        "next_action": [
            "Review individual_scores.csv and confirm 040 recomputed OOF CV",
            "Review pairwise_oof_correlation_focus_040.csv and pairwise_oof_correlation_focus_039.csv",
            "Review safe_blend_diagnostics_focus_040.csv and safe_blend_diagnostics_focus_039.csv",
            "Submit only best safe blend, not in-sample meta rows",
            "Keep 040 as slot_1 unless 041 best safe blend confirms on Public LB",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)


Saved files:
 - best_blend_info.json
 - blend_add039_gpu040_summary.json
 - blend_diagnostics.csv
 - individual_scores.csv
 - memo.yml
 - oof_exp_20260611_041_blend_add039_gpu040_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pairwise_oof_correlation_focus_028.csv
 - pairwise_oof_correlation_focus_031.csv
 - pairwise_oof_correlation_focus_033.csv
 - pairwise_oof_correlation_focus_036_blend.csv
 - pairwise_oof_correlation_focus_036_gpu.csv
 - pairwise_oof_correlation_focus_037.csv
 - pairwise_oof_correlation_focus_038.csv
 - pairwise_oof_correlation_focus_039.csv
 - pairwise_oof_correlation_focus_040.csv
 - pred_exp_20260611_041_blend_add039_gpu040_check_best_blend_proba.npy
 - role_judgement.json
 - safe_blend_diagnostics.csv
 - safe_blend_diagnostics_focus_028.csv
 - safe_blend_diagnostics_focus_029.csv
 - safe_blend_diagnostics_focus_030.csv
 - safe_blend_diagnostics_focus_031.csv
 - safe_blend_diagnostics_focus_032.csv
 - safe_blend_diagnostics_focus_033.csv
 - safe_b